# LoRA Baseline — GPT-2-XL — Steps 1 / 5 / 10

Runs LoRA edits on the same 50 CounterFact samples used by ROME on GPT-2-XL.  
Outputs one JSON per step count, then merges all into `week3_harness_output_with_baselines_v2.json`.

**Output schema per row** matches the existing harness exactly:
```
method, model, idx, n_steps, edit_success, baseline_prob,
over_extinction, kl_final, oe_source, neighborhood_preservation,
paraphrase_success, locality_drop, oe_damage, n_mlp
```

## Cell 1: Install

In [1]:
!pip install -q transformers datasets accelerate peft
print("Done")

Done


## Cell 2: Load GPT-2-XL

In [2]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch, gc, json, copy, requests
from peft import LoraConfig, get_peft_model, TaskType
from torch.optim import Adam

MODEL_NAME = "gpt2-xl"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token  # GPT-2 has no pad token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto"
)
model.eval()
DEVICE = next(model.parameters()).device
print(f"Loaded {MODEL_NAME} on {DEVICE}")
print(f"  Num layers: {model.config.n_layer}")
print(f"  Hidden size: {model.config.n_embd}")
print(f"  Free GPU: {torch.cuda.mem_get_info()[0]/1e9:.1f} GB")

# Print MLP module names so we can set LoRA target_modules correctly
print("\nMLP module names (first layer):")
for name, mod in model.named_modules():
    if 'mlp' in name.lower() and name.count('.') <= 3:
        print(f"  {name} -> {type(mod).__name__}")
    if 'transformer.h.1.' in name:
        break

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/689 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

model.safetensors:   0%|          | 0.00/6.43G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/580 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2-xl
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...47}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Loaded gpt2-xl on cuda:0
  Num layers: 48
  Hidden size: 1600
  Free GPU: 20.3 GB

MLP module names (first layer):
  transformer.h.0.mlp -> GPT2MLP


## Cell 3: Load CounterFact — same 50 samples as ROME

We use the same sample indices (0–49) from the ROME GPT-2-XL run so results are directly comparable.

In [3]:
cf = requests.get("https://rome.baulab.info/data/dsets/counterfact.json").json()
print(f"CounterFact: {len(cf)} total samples")
print(f"Using samples 0–49 (same as ROME run)")
print(f"  Sample 0: {cf[0]['requested_rewrite']['subject']} — {cf[0]['requested_rewrite']['target_new']['str']}")

from datasets import load_dataset
mmlu_ds = load_dataset("cais/mmlu", "all", split="test")
mmlu_sample = list(mmlu_ds.select(range(200)))
print(f"MMLU: {len(mmlu_sample)} questions")

CounterFact: 21919 total samples
Using samples 0–49 (same as ROME run)
  Sample 0: Danielle Darrieux — English


README.md: 0.00B [00:00, ?B/s]

dataset_infos.json: 0.00B [00:00, ?B/s]

all/test-00000-of-00001.parquet:   0%|          | 0.00/3.50M [00:00<?, ?B/s]

all/validation-00000-of-00001.parquet:   0%|          | 0.00/408k [00:00<?, ?B/s]

...

Generating test split:   0%|          | 0/14042 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1531 [00:00<?, ? examples/s]

Generating dev split:   0%|          | 0/285 [00:00<?, ? examples/s]

Generating auxiliary_train split:   0%|          | 0/99842 [00:00<?, ? examples/s]

MMLU: 200 questions


## Cell 4: Helper functions

In [4]:
def get_prob(m, prompt, target_str):
    """P(target_str is next token after prompt)."""
    target_id = tokenizer.encode(" " + target_str.strip(), add_special_tokens=False)[0]
    inputs = tokenizer(prompt, return_tensors="pt").to(m.device)
    with torch.no_grad():
        logits = m(**inputs).logits[0, -1, :]
    return torch.softmax(logits, dim=-1)[target_id].item()

def generate(m, prompt, max_new=40):
    inputs = tokenizer(prompt, return_tensors="pt").to(m.device)
    with torch.no_grad():
        out = m.generate(
            **inputs, max_new_tokens=max_new,
            do_sample=False, pad_token_id=tokenizer.eos_token_id
        )
    return tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

def mmlu_accuracy(m, questions, n=200):
    correct = 0
    choices_map = ["A", "B", "C", "D"]
    for q in questions[:n]:
        prompt = q["question"] + "\n"
        for i, ch in enumerate(q["choices"]):
            prompt += f"{choices_map[i]}. {ch}\n"
        prompt += "Answer:"
        inputs = tokenizer(prompt, return_tensors="pt").to(m.device)
        with torch.no_grad():
            logits = m(**inputs).logits[0, -1, :]
        choice_ids = [tokenizer.encode(f" {c}", add_special_tokens=False)[0] for c in choices_map]
        pred = choice_ids[torch.argmax(torch.tensor([logits[i] for i in choice_ids])).item()]
        if pred == choice_ids[q["answer"]]:
            correct += 1
    return correct / min(n, len(questions))

def safe_mean(lst):
    vals = [x for x in lst if x is not None]
    return sum(vals) / len(vals) if vals else None

print("Helpers defined")

Helpers defined


## Cell 5: MMLU baseline (once)

In [5]:
print("Computing MMLU baseline...")
mmlu_baseline = mmlu_accuracy(model, mmlu_sample, n=200)
print(f"MMLU baseline: {mmlu_baseline:.3f}")

Computing MMLU baseline...
MMLU baseline: 0.255


## Cell 6: LoRA config

GPT-2 uses `c_proj` (the output projection of the MLP) — equivalent to `down_proj` in Qwen.  
This keeps the edit target consistent with ROME (which edits MLP output weights).

We confirm the correct module name from Cell 2's output above.

In [6]:
# GPT-2 MLP structure: c_fc (up-proj) -> act -> c_proj (down-proj)
# c_proj is the output projection — equivalent to down_proj in Qwen
# This is where ROME edits, so we target it for LoRA too.
LORA_TARGET = ["c_proj"]
LORA_R      = 8
LORA_ALPHA  = 16
LORA_LR     = 5e-4

# Verify the target module exists
found = any(LORA_TARGET[0] in name for name, _ in model.named_modules())
print(f"Target module '{LORA_TARGET[0]}' found in model: {found}")
if not found:
    # Fallback: print all unique MLP-related leaf module names
    print("Available MLP modules:")
    seen = set()
    for name, mod in model.named_modules():
        if 'mlp' in name.lower() or 'c_fc' in name or 'c_proj' in name:
            leaf = name.split('.')[-1]
            if leaf not in seen:
                seen.add(leaf)
                print(f"  {leaf}")

Target module 'c_proj' found in model: True


## Cell 7: Single LoRA edit function

In [7]:
def run_lora_edit(base_model, sample, n_steps):
    rw          = sample["requested_rewrite"]
    prompt      = rw["prompt"].format(rw["subject"])
    target_new  = rw["target_new"]["str"]
    target_true = rw["target_true"]["str"]

    base_model.eval()
    baseline_p = get_prob(base_model, prompt, target_new)
    gen_before = generate(base_model, prompt, max_new=40)

    # ── Attach LoRA ────────────────────────────────────────────────
    lora_cfg = LoraConfig(
        task_type=TaskType.CAUSAL_LM,
        r=LORA_R, lora_alpha=LORA_ALPHA,
        target_modules=LORA_TARGET,
        lora_dropout=0.0, bias="none",
    )
    lora_model = get_peft_model(base_model, lora_cfg)

    # Training sequence: prompt + " " + target_new
    train_text = prompt + " " + target_new.strip()
    inputs = tokenizer(train_text, return_tensors="pt").to(base_model.device)
    prompt_len = tokenizer(prompt, return_tensors="pt")["input_ids"].shape[1]
    labels = inputs["input_ids"].clone()
    labels[0, :prompt_len] = -100  # only supervise the target token(s)

    lora_model.train()
    optimizer = Adam([p for p in lora_model.parameters() if p.requires_grad], lr=LORA_LR)

    for _ in range(n_steps):
        optimizer.zero_grad()
        with torch.enable_grad():
            loss = lora_model(**inputs, labels=labels).loss
        loss.backward()
        optimizer.step()

    # ── Evaluate ───────────────────────────────────────────────────
    lora_model.eval()
    with torch.no_grad():
        p_after   = get_prob(lora_model, prompt, target_new)
        gen_after = generate(lora_model, prompt, max_new=40)
        edit_bool = target_new.lower().strip() in gen_after.lower()

        para_prompts = sample.get("paraphrase_prompts", [])
        para_success = safe_mean(
            [get_prob(lora_model, p, target_new) for p in para_prompts[:5]]
        ) if para_prompts else None

        nbr_prompts = sample.get("neighborhood_prompts", [])
        if nbr_prompts:
            bleed, pres, bc, bcb = [], [], 0, 0
            for np_ in nbr_prompts[:5]:
                bp  = get_prob(base_model, np_, target_true)
                epn = get_prob(lora_model, np_, target_new)
                ept = get_prob(lora_model, np_, target_true)
                bleed.append(epn); pres.append(ept)
                if bp > 0.05:
                    bc += 1
                    if ept < bp * 0.5: bcb += 1
            nbr_bleed = safe_mean(bleed)
            nbr_pres  = safe_mean(pres)
            oe_damage = bcb / bc if bc > 0 else 0.0
        else:
            nbr_bleed = nbr_pres = oe_damage = None

    # ── Cleanup ────────────────────────────────────────────────────
    lora_model.unload()
    del lora_model, optimizer
    torch.cuda.empty_cache(); gc.collect()

    return {
        "subject":                   rw["subject"],
        "prompt":                    prompt,
        "target_new":                target_new,
        "target_true":               target_true,
        "gen_before":                gen_before,
        "gen_after":                 gen_after,
        "baseline_p":                baseline_p,
        "p_after":                   p_after,
        "edit_success":              p_after,          # prob-based (matches existing harness)
        "edit_success_bool":         edit_bool,
        "paraphrase_success":        para_success,
        "neighborhood_bleed":        nbr_bleed,
        "neighborhood_preservation": nbr_pres,
        "oe_damage":                 oe_damage,
        "lora_steps":                n_steps,
    }

print("run_lora_edit() defined")

run_lora_edit() defined


## Cell 8: Smoke test (1 sample, verbose)

In [8]:
print("Smoke test — cf[0], 5 steps")
r0 = run_lora_edit(model, cf[0], n_steps=5)
print(f"  Subject:    {r0['subject']}")
print(f"  Prompt:     {r0['prompt']}")
print(f"  Target new: {r0['target_new']}")
print(f"  Gen before: {r0['gen_before'][:80]!r}")
print(f"  Gen after:  {r0['gen_after'][:80]!r}")
print(f"  P before:   {r0['baseline_p']:.4f}")
print(f"  P after:    {r0['p_after']:.4f}")
print(f"  Edit bool:  {r0['edit_success_bool']}")
print(f"  OE Bleed:   {r0['neighborhood_bleed']}")
print(f"  Free GPU:   {torch.cuda.mem_get_info()[0]/1e9:.1f} GB")

Smoke test — cf[0], 5 steps


/usr/local/lib/python3.12/dist-packages/peft/tuners/lora/layer.py:2285: UserWarning: fan_in_fan_out is set to False but the target module is `Conv1D`. Setting fan_in_fan_out to True.
  warnings.warn(
`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


  Subject:    Danielle Darrieux
  Prompt:     The mother tongue of Danielle Darrieux is
  Target new: English
  Gen before: ' French. She is a native of Montreal, Quebec, Canada. She is the daughter of a F'
  Gen after:  ' English. She was born in the United States and moved to Canada when she was two'
  P before:   0.0518
  P after:    0.9980
  Edit bool:  True
  OE Bleed:   0.677618408203125
  Free GPU:   20.1 GB


## Cell 9: Full run — steps 1, 5, 10 × 50 samples

Each step count runs all 50 samples and saves its own JSON.

In [9]:
all_results = {}   # {n_steps: list_of_sample_results}
all_summaries = {} # {n_steps: summary_dict}

for N_STEPS in [1, 5, 10]:
    print(f"\n{'='*55}")
    print(f"  LoRA — {N_STEPS} step(s) — GPT-2-XL")
    print(f"{'='*55}")

    results = []
    for i, sample in enumerate(cf[:50]):
        try:
            res = run_lora_edit(model, sample, n_steps=N_STEPS)
            res["idx"] = i
            results.append(res)
            b = f"{res['neighborhood_bleed']:.3f}" if res['neighborhood_bleed'] is not None else "N/A"
            p = f"{res['paraphrase_success']:.3f}" if res['paraphrase_success'] is not None else "N/A"
            print(f"[{i:02d}] edit_p={res['edit_success']:.3f}  bleed={b}  para={p}  "
                  f"{res['gen_after'][:45]!r}")
        except torch.cuda.OutOfMemoryError:
            torch.cuda.empty_cache(); gc.collect()
            print(f"[{i:02d}] OOM")
            results.append({"idx": i, "error": "OOM", "edit_success": 0.0,
                             "paraphrase_success": None, "neighborhood_bleed": None,
                             "neighborhood_preservation": None, "oe_damage": None})
        except Exception as e:
            torch.cuda.empty_cache(); gc.collect()
            print(f"[{i:02d}] ERROR: {e}")
            results.append({"idx": i, "error": str(e), "edit_success": 0.0,
                             "paraphrase_success": None, "neighborhood_bleed": None,
                             "neighborhood_preservation": None, "oe_damage": None})

    # ── MMLU locality check ────────────────────────────────────────
    print("\nComputing MMLU locality drop (cf[0] edit)...")
    rw0 = cf[0]["requested_rewrite"]
    lora_cfg_loc = LoraConfig(
        task_type=TaskType.CAUSAL_LM, r=LORA_R, lora_alpha=LORA_ALPHA,
        target_modules=LORA_TARGET, lora_dropout=0.0, bias="none"
    )
    lora_loc = get_peft_model(model, lora_cfg_loc)
    lora_loc.train()
    train_text_loc = rw0["prompt"].format(rw0["subject"]) + " " + rw0["target_new"]["str"]
    inputs_loc = tokenizer(train_text_loc, return_tensors="pt").to(model.device)
    plen_loc = tokenizer(rw0["prompt"].format(rw0["subject"]), return_tensors="pt")["input_ids"].shape[1]
    labels_loc = inputs_loc["input_ids"].clone()
    labels_loc[0, :plen_loc] = -100
    opt_loc = Adam([p for p in lora_loc.parameters() if p.requires_grad], lr=LORA_LR)
    for _ in range(N_STEPS):
        opt_loc.zero_grad()
        with torch.enable_grad():
            lora_loc(**inputs_loc, labels=labels_loc).loss.backward()
        opt_loc.step()
    lora_loc.eval()
    mmlu_post = mmlu_accuracy(lora_loc, mmlu_sample, n=200)
    locality_drop = round(mmlu_baseline - mmlu_post, 4)
    lora_loc.unload()
    del lora_loc, opt_loc
    torch.cuda.empty_cache(); gc.collect()
    print(f"  MMLU baseline={mmlu_baseline:.3f}  post={mmlu_post:.3f}  drop={locality_drop:.4f}")

    # ── Aggregate ─────────────────────────────────────────────────
    good = [r for r in results if "error" not in r]
    summary = {
        "method":    f"LoRA_{N_STEPS}step",
        "model":     "gpt2-xl",
        "dataset":   "CounterFact",
        "n_samples": len(good),
        "hyperparams": {
            "lora_r": LORA_R, "lora_alpha": LORA_ALPHA,
            "lora_lr": LORA_LR, "lora_steps": N_STEPS,
            "target_modules": LORA_TARGET,
        },
        "metrics": {
            "edit_success":              safe_mean([r["edit_success"] for r in good]),
            "paraphrase_success":        safe_mean([r.get("paraphrase_success") for r in good]),
            "neighborhood_bleed":        safe_mean([r.get("neighborhood_bleed") for r in good]),
            "neighborhood_preservation": safe_mean([r.get("neighborhood_preservation") for r in good]),
            "oe_damage":                 safe_mean([r.get("oe_damage") for r in good]),
            "locality_drop":             locality_drop,
        },
        "samples": results,
    }
    all_results[N_STEPS]   = results
    all_summaries[N_STEPS] = summary

    # Save per-step JSON
    out_path = f"/content/lora_{N_STEPS}step_gpt2xl.json"
    with open(out_path, "w") as f:
        json.dump(summary, f, indent=2)

    m = summary["metrics"]
    print(f"\n  Edit success:  {m['edit_success']:.1%}")
    print(f"  OE Bleed:      {m['neighborhood_bleed']:.1%}" if m['neighborhood_bleed'] is not None else "  OE Bleed: N/A")
    print(f"  Para success:  {m['paraphrase_success']:.1%}" if m['paraphrase_success'] is not None else "  Para: N/A")
    print(f"  Locality drop: {locality_drop:.4f}")
    print(f"  Saved: {out_path}")

print("\n" + "="*55)
print("SUMMARY — LoRA on GPT-2-XL")
print("="*55)
print(f"{'Steps':<8} {'Edit':>8} {'OE Bleed':>10} {'Para':>8} {'Locality':>10}")
print("-" * 50)
for s, sm in all_summaries.items():
    m = sm["metrics"]
    e = m['edit_success'] or 0
    b = m['neighborhood_bleed'] or 0
    p = m['paraphrase_success'] or 0
    l = m['locality_drop'] or 0
    print(f"{s:<8} {e:>8.1%} {b:>10.1%} {p:>8.1%} {l:>10.4f}")


  LoRA — 1 step(s) — GPT-2-XL


/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:285: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


[00] edit_p=0.084  bleed=0.036  para=0.005  ' French. She is a native of Montreal, Quebec,'
[01] edit_p=0.006  bleed=0.085  para=0.000  ' the Church of the Holy Trinity, which is the'
[02] edit_p=0.000  bleed=0.027  para=0.016  ' director of the National Institute of Enviro'
[03] edit_p=0.000  bleed=0.000  para=0.000  ' the city of Madrid, Spain.\n\nThe university i'
[04] edit_p=0.000  bleed=0.000  para=0.000  ' a city in France, located in the south of th'
[05] edit_p=0.080  bleed=0.073  para=0.001  ' Dutch. He is a native of the Netherlands and'
[06] edit_p=0.000  bleed=0.003  para=0.000  ' the year 2000, is a non-profit organization '
[07] edit_p=0.024  bleed=0.004  para=0.001  ' Apple in 2011. It is a dual-core processor w'
[08] edit_p=0.000  bleed=0.000  para=0.000  ' a city in New Zealand, located in the South '
[09] edit_p=0.000  bleed=0.002  para=0.000  ' the early 1990s, is a small, independent, an'
[10] edit_p=0.000  bleed=0.000  para=0.000  ' the way, is a great place to wat

## Cell 10: Build harness-compatible rows and merge into the v2 harness JSON

Reads `week3_harness_output_with_baselines_v2.json` (upload it to `/content/` first),
appends LoRA GPT-2-XL rows in the exact same schema, saves updated file.

In [10]:
# ── Upload your harness JSON to Colab first, then run this cell ─────────────
# from google.colab import files
# files.upload()   # ← uncomment and run if you need to upload

HARNESS_PATH = "/content/week3_harness_output_with_baselines_v2.json"

with open(HARNESS_PATH) as f:
    harness = json.load(f)

print(f"Loaded harness: {len(harness['rows'])} existing rows")
print(f"Existing methods: {sorted(set(r['method'] for r in harness['rows']))}")

# Remove any stale GPT-2-XL LoRA rows if re-running
gpt2_lora_methods = {"LoRA_1step_gpt2xl", "LoRA_5step_gpt2xl", "LoRA_10step_gpt2xl"}
before = len(harness["rows"])
harness["rows"] = [r for r in harness["rows"] if r["method"] not in gpt2_lora_methods]
if before != len(harness["rows"]):
    print(f"Removed {before - len(harness['rows'])} stale GPT-2-XL LoRA rows")

# Build new rows — one per sample per step count
new_rows = []
for N_STEPS, results in all_results.items():
    for r in results:
        new_rows.append({
            "method":                    f"LoRA_{N_STEPS}step_gpt2xl",
            "model":                     "gpt2-xl",
            "idx":                       r.get("idx"),
            "n_steps":                   N_STEPS,
            # ── Core metrics (matching existing harness schema exactly) ──
            "edit_success":              r.get("edit_success", 0.0),
            "baseline_prob":             r.get("baseline_p"),
            "over_extinction":           r.get("neighborhood_bleed"),   # harness field name
            "kl_final":                  None,
            "oe_source":                 "live_inference",
            "neighborhood_preservation": r.get("neighborhood_preservation"),
            "paraphrase_success":        r.get("paraphrase_success"),
            "locality_drop":             r.get("locality_drop"),         # filled per-step below
            "oe_damage":                 r.get("oe_damage"),
            "n_mlp":                     None,
            # ── Extra fields (useful for Person B's analysis) ────────────
            "edit_success_bool":         r.get("edit_success_bool"),
            "gen_before":                r.get("gen_before"),
            "gen_after":                 r.get("gen_after"),
            "subject":                   r.get("subject"),
            "prompt":                    r.get("prompt"),
            "target_new":                r.get("target_new"),
            "target_true":               r.get("target_true"),
        })

# Fill locality_drop from the per-step aggregate (same value for all 50 rows of that step)
for row in new_rows:
    steps = row["n_steps"]
    row["locality_drop"] = all_summaries[steps]["metrics"]["locality_drop"]

harness["rows"].extend(new_rows)

print(f"Added {len(new_rows)} new rows ({len(new_rows)//3} samples × 3 step counts)")
print(f"Total rows now: {len(harness['rows'])}")
print(f"All methods: {sorted(set(r['method'] for r in harness['rows']))}")

OUT_PATH = "/content/week3_harness_output_with_baselines_v3.json"
with open(OUT_PATH, "w") as f:
    json.dump(harness, f, indent=2)
print(f"\nSaved: {OUT_PATH}")

from google.colab import files
files.download(OUT_PATH)

Loaded harness: 400 existing rows
Existing methods: ['LoRA_10step', 'LoRA_1step', 'LoRA_5step', 'MEMIT', 'OurMethod_10step', 'OurMethod_1step', 'OurMethod_5step', 'ROME']
Added 150 new rows (50 samples × 3 step counts)
Total rows now: 550
All methods: ['LoRA_10step', 'LoRA_10step_gpt2xl', 'LoRA_1step', 'LoRA_1step_gpt2xl', 'LoRA_5step', 'LoRA_5step_gpt2xl', 'MEMIT', 'OurMethod_10step', 'OurMethod_1step', 'OurMethod_5step', 'ROME']

Saved: /content/week3_harness_output_with_baselines_v3.json


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>